# Evaluación de modelos no supervisados

Compara Average Precision, Precision@k y Recall@k para:
- Gaussian (Mahalanobis)
- Isolation Forest
- CUSUM (cambio estructural)
- ETS (series de tiempo)

## Setup

In [48]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('../src').resolve()))

import pandas as pd

from metricas_no_supervisado import resumen_metricas, average_precision

PATH_RES  = '../data/Resultados/'
PATH_VISITAS  = '../data/TopVisitar/'
PATH_DATA = '../data/EMCALI/datos_procesados/'

TOP_K = 5000  # Número máximo de clientes a considerar por método

## Ground truth

Solo se evalúa sobre clientes que fueron inspeccionados en campo.

In [12]:
gt = pd.read_csv(PATH_DATA + 'inspecciones_etiquetadas.csv')

inspeccionados  = set(gt['producto'].unique())
anomalos_reales = set(gt[gt['anomalo'] == 1]['producto'].unique())

print(f'Clientes inspeccionados: {len(inspeccionados):,}')
print(f'Anomalías reales:        {len(anomalos_reales):,}')
print(f'Tasa base anomalías:     {len(anomalos_reales)/len(inspeccionados):.1%}')


Clientes inspeccionados: 13,287
Anomalías reales:        5,778
Tasa base anomalías:     43.5%


## Función auxiliar

In [13]:
def evaluar_modelo(ranking_ids, nombre, ks=None):
    """
    Filtra el ranking a clientes inspeccionados, toma los primeros TOP_K,
    y calcula métricas. El denominador del recall son los anómalos dentro
    de esos TOP_K inspeccionados.

    ranking_ids: lista/array de CLIENTE_ID ordenados de más a menos anómalo
                 (ranking completo del modelo).
    """
    inspeccionados_rankeados = [x for x in ranking_ids if x in inspeccionados][:TOP_K]
    anomalos_en_universo = {x for x in inspeccionados_rankeados if x in anomalos_reales}

    ap = average_precision(inspeccionados_rankeados, anomalos_en_universo)
    resumen = resumen_metricas(inspeccionados_rankeados, anomalos_en_universo, ks=ks)
    resumen.insert(0, 'modelo', nombre)

    print(f'[{nombre}] inspeccionados en top{TOP_K}: {len(inspeccionados_rankeados):,} | anómalos: {len(anomalos_en_universo):,} | AP = {ap:.4f}')
    return resumen, inspeccionados_rankeados

## Cargar resultados de cada modelo

In [14]:
gaussian = pd.read_csv(PATH_RES + 'gaussian_resultados_final.csv')
iso_forest = pd.read_csv(PATH_RES + 'if_resultados_final.csv')
individuales = pd.read_csv(PATH_RES + 'results_cliente_metodos_individuales.csv')

print('Gaussian:      ', gaussian.shape)
print('Isolation Forest:', iso_forest.shape)
print('Individuales:  ', individuales.shape)

Gaussian:       (592026, 11)
Isolation Forest: (590878, 137)
Individuales:   (1029768, 10)


In [15]:
(1-individuales.rank_ts.isna()).sum()

13618

## Gaussian (Mahalanobis)

In [16]:
ranking_gaussian = (gaussian
    .sort_values('rank_gaussian', ascending=True)
    ['CLIENTE_ID'].values)

resumen_gaussian, retrieved_gaussian = evaluar_modelo(ranking_gaussian, 'Gaussian')
resumen_gaussian

[Gaussian] inspeccionados en top5000: 5,000 | anómalos: 2,034 | AP = 0.4217


,modelo,k,precision_at_k,recall_at_k,ap
0,Gaussian,50,0.540000,0.013274,0.42171
1,Gaussian,100,0.480000,0.023599,0.42171
2,Gaussian,500,0.440000,0.108161,0.42171
3,Gaussian,1000,0.424000,0.208456,0.42171
4,Gaussian,2000,0.417000,0.410029,0.42171
5,Gaussian,3000,0.414667,0.611603,0.42171
6,Gaussian,4000,0.412000,0.810226,0.42171
7,Gaussian,5000,0.406800,1.000000,0.42171
8,Gaussian,5000,0.406800,1.000000,0.42171


## Isolation Forest

In [17]:
ranking_if = (iso_forest
    .sort_values('anomaly_score_scaled', ascending=False)
    ['CLIENTE_ID'].values)

resumen_if, retrieved_if = evaluar_modelo(ranking_if, 'Isolation Forest')
resumen_if

[Isolation Forest] inspeccionados en top5000: 5,000 | anómalos: 2,185 | AP = 0.4375


,modelo,k,precision_at_k,recall_at_k,ap
0,Isolation Forest,50,0.38000,0.008696,0.437521
1,Isolation Forest,100,0.41000,0.018764,0.437521
2,Isolation Forest,500,0.42200,0.096568,0.437521
3,Isolation Forest,1000,0.44100,0.201831,0.437521
4,Isolation Forest,2000,0.44350,0.405950,0.437521
5,Isolation Forest,3000,0.43500,0.597254,0.437521
6,Isolation Forest,4000,0.44075,0.806865,0.437521
7,Isolation Forest,5000,0.43700,1.000000,0.437521
8,Isolation Forest,5000,0.43700,1.000000,0.437521


## CUSUM (cambio estructural)

In [18]:
cusum = (individuales
    .dropna(subset=['rank_cambio_estructural'])
    .sort_values('rank_cambio_estructural', ascending=True))

ranking_cusum = cusum['CLIENTE_ID'].values
resumen_cusum, retrieved_cusum = evaluar_modelo(ranking_cusum, 'CUSUM')
resumen_cusum

[CUSUM] inspeccionados en top5000: 5,000 | anómalos: 2,056 | AP = 0.3938


,modelo,k,precision_at_k,recall_at_k,ap
0,CUSUM,50,0.240000,0.005837,0.393798
1,CUSUM,100,0.280000,0.013619,0.393798
2,CUSUM,500,0.370000,0.089981,0.393798
3,CUSUM,1000,0.389000,0.189202,0.393798
4,CUSUM,2000,0.398500,0.387646,0.393798
5,CUSUM,3000,0.401667,0.586089,0.393798
6,CUSUM,4000,0.408750,0.795233,0.393798
7,CUSUM,5000,0.411200,1.000000,0.393798
8,CUSUM,5000,0.411200,1.000000,0.393798


## ETS (series de tiempo)

In [19]:
ets = (individuales
    .dropna(subset=['rank_ts'])
    .sort_values('rank_ts', ascending=True))

ranking_ets = ets['CLIENTE_ID'].values
resumen_ets, retrieved_ets = evaluar_modelo(ranking_ets, 'ETS')
resumen_ets

[ETS] inspeccionados en top5000: 299 | anómalos: 125 | AP = 0.4053


,modelo,k,precision_at_k,recall_at_k,ap
0,ETS,50,0.42000,0.168,0.405284
1,ETS,100,0.41000,0.328,0.405284
2,ETS,299,0.41806,1.000,0.405284


## Tabla comparativa

In [20]:
tabla = pd.concat([resumen_gaussian, resumen_if, resumen_cusum, resumen_ets], ignore_index=True)
tabla


,modelo,k,precision_at_k,recall_at_k,ap
0,Gaussian,50,0.540000,0.013274,0.421710
1,Gaussian,100,0.480000,0.023599,0.421710
2,Gaussian,500,0.440000,0.108161,0.421710
3,Gaussian,1000,0.424000,0.208456,0.421710
4,Gaussian,2000,0.417000,0.410029,0.421710
5,Gaussian,3000,0.414667,0.611603,0.421710
6,Gaussian,4000,0.412000,0.810226,0.421710
7,Gaussian,5000,0.406800,1.000000,0.421710
8,Gaussian,5000,0.406800,1.000000,0.421710
9,Isolation Forest,50,0.380000,0.008696,0.437521


# Sacar Top 5000

In [41]:
isolation_forest = pd.read_csv(PATH_RES + 'if_resultados_final.csv')
resultados_supervisados = pd.read_csv(PATH_RES + 'supervisado_predicciones_completo.csv')

In [42]:
resultados_supervisados = resultados_supervisados.sort_values('prediccion_lgbm', ascending=False).iloc[:TOP_K]
isolation_forest = isolation_forest.sort_values("anomaly_score", ascending=True).iloc[:TOP_K]

In [49]:
isolation_forest.rename(columns = {'CLIENTE_ID': 'producto'}, inplace = True)

In [50]:
len(set(resultados_supervisados.producto).intersection(isolation_forest.producto))

49

In [54]:
resultados_supervisados[['producto', 'prediccion_lgbm']].to_excel(PATH_VISITAS + 'top_5000_supervisado.xlsx')
isolation_forest[['producto', 'anomaly_score']].to_excel(PATH_VISITAS + 'top_5000_no_supervisado.xlsx')